# 09 - End-to-End Demo

Provide a compact, presentation-ready walkthrough of the MovieLens-1M
cold-start pipeline. The notebook consumes frozen protocol and report bundles;
it does not train models, tune thresholds, or rebuild the dataset. If PyTorch
Geometric is installed, the graph sample is materialized as `HeteroData`; if it
is not installed, the same sample is exported as PyG-compatible tensors and the
demo is marked `WARN` rather than pretending PyG executed.

## Linkage and Cell Plan

**Upstream links:** notebook 08 provides the final report bundle at
`reports/ml-1m/results-v1/manifest.json`; notebook 02 provides the verified
protocol bundle at `protocols/ml-1m/coldstart-v1/manifest.json` for a small live
data/graph walkthrough.

**Cell work:**
1. Verify/load notebook 08 report outputs and notebook 02 protocol tables.
2. Summarize the cold-start dataset contract and phase support visibility.
3. Build one bounded graph sample in PyG form when available.
4. Show model roles and final result evidence from notebook 08.
5. Export a concise demo bundle and final checks.

In [ ]:
from __future__ import annotations

import hashlib
import importlib.util
import json
import os
import platform
import shutil
import uuid
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable

REQUIRED_PACKAGES = ["numpy", "pandas", "IPython"]
MISSING_PACKAGES = [
    package for package in REQUIRED_PACKAGES if importlib.util.find_spec(package) is None
]
if MISSING_PACKAGES:
    raise RuntimeError(
        "Notebook 09 requires these packages in the active kernel: "
        + ", ".join(MISSING_PACKAGES)
        + ". Run it in the project ML/Kaggle environment used for demo notebooks."
    )

import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

TORCH_AVAILABLE = importlib.util.find_spec("torch") is not None
PYG_AVAILABLE = importlib.util.find_spec("torch_geometric") is not None
if TORCH_AVAILABLE:
    import torch
else:
    torch = None
if PYG_AVAILABLE:
    from torch_geometric.data import HeteroData
else:
    HeteroData = None


def show_records(records: Iterable[dict[str, Any]]) -> None:
    display(pd.DataFrame(list(records)))


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def reject_json_constant(value: str) -> None:
    raise ValueError(f"Non-finite JSON constant is not allowed: {value}")


def strict_json_loads(payload: str | bytes) -> Any:
    return json.loads(payload, parse_constant=reject_json_constant)


def strict_json_dumps(value: Any, **kwargs: Any) -> str:
    return json.dumps(value, allow_nan=False, **kwargs)


def project_root(start: Path) -> Path:
    override = os.environ.get("COLDSTART_PROJECT_ROOT")
    if override:
        return Path(override).expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists():
            return candidate.resolve()
    return start.resolve()


def resolve_inside(root: Path, relative_path: str) -> Path:
    resolved_root = root.resolve()
    resolved = (resolved_root / relative_path).resolve()
    resolved.relative_to(resolved_root)
    return resolved


def load_json(path: Path) -> dict[str, Any]:
    value = strict_json_loads(path.read_text(encoding="utf-8"))
    if not isinstance(value, dict):
        raise ValueError(f"JSON document is not an object: {path}")
    return value


EXECUTION_CONTEXT = "kaggle" if Path("/kaggle/input").exists() else "local"
PROJECT_ROOT = project_root(Path.cwd())
WORKSPACE_ROOT = Path(
    os.environ.get(
        "COLDSTART_WORKSPACE_ROOT",
        "/kaggle/working" if EXECUTION_CONTEXT == "kaggle" else PROJECT_ROOT / ".notebook",
    )
).expanduser().resolve()
INPUT_ROOT = Path(
    os.environ.get(
        "COLDSTART_INPUT_ROOT",
        "/kaggle/input" if EXECUTION_CONTEXT == "kaggle" else PROJECT_ROOT / "data",
    )
).expanduser().resolve()
ARTIFACT_ROOT = Path(
    os.environ.get("COLDSTART_ARTIFACT_ROOT", WORKSPACE_ROOT / "artifacts")
).expanduser().resolve()
REPORT_RELATIVE_MANIFEST = Path("reports/ml-1m/results-v1/manifest.json")
PROTOCOL_RELATIVE_MANIFEST = Path("protocols/ml-1m/coldstart-v1/manifest.json")
OUTPUT_ROOT = ARTIFACT_ROOT / "demos" / "ml-1m" / "end-to-end-v1"
PHASES = ["Cold", "Warm A", "Warm B", "Warm C"]


def artifact_candidates(relative_manifest: Path, terminal_name: str) -> list[tuple[Path, Path]]:
    candidates: list[tuple[Path, Path]] = []
    roots = [PROJECT_ROOT / ".notebook" / "artifacts", ARTIFACT_ROOT]
    if INPUT_ROOT.is_dir():
        roots.append(INPUT_ROOT)
    for root in roots:
        pointer = root / relative_manifest
        if pointer.is_file():
            candidates.append((root.resolve(), pointer.resolve()))
        for direct_root in sorted(root.rglob(terminal_name)) if root.is_dir() else []:
            direct = direct_root / "manifest.json"
            if direct_root.name == terminal_name and direct.is_file():
                candidates.append((direct_root.parents[2].resolve(), direct.resolve()))
    unique: list[tuple[Path, Path]] = []
    seen: set[str] = set()
    for root, pointer in candidates:
        key = str(pointer)
        if key not in seen:
            seen.add(key)
            unique.append((root, pointer))
    return unique


def read_csv_artifact(root: Path, manifest: dict[str, Any], name: str) -> pd.DataFrame:
    artifact = manifest["artifacts"][name]
    schema = manifest["output_schemas"][name]
    path = resolve_inside(root, artifact["path"])
    if not path.is_file() or sha256_file(path) != artifact["sha256"]:
        raise ValueError(f"Artifact hash verification failed: {name}")
    table = pd.read_csv(path, dtype=schema["read_csv_dtypes"])
    if list(table.columns) != schema["columns"] or len(table) != artifact["rows"]:
        raise ValueError(f"Artifact schema/row contract failed: {name}")
    return table


def load_verified_report(root: Path, pointer: Path) -> tuple[dict[str, Any], dict[str, pd.DataFrame], dict[str, Path]]:
    pointer_bytes = pointer.read_bytes()
    manifest = strict_json_loads(pointer_bytes)
    if manifest.get("report_schema_version") != "results-report-v1":
        raise ValueError("Unexpected notebook 08 report schema")
    if manifest.get("report_status") not in {"PASS", "WARN"}:
        raise ValueError(f"Notebook 08 report status is not PASS/WARN: {manifest.get('report_status')!r}")
    checks = manifest.get("checks")
    if not isinstance(checks, list) or not checks or any(row.get("status") == "ERROR" for row in checks if isinstance(row, dict)):
        raise ValueError("Notebook 08 report has missing or blocking checks")
    bundle_manifest = resolve_inside(root, manifest["bundle_manifest"])
    expected_bundle_manifest = (pointer.parent / "generations" / manifest["bundle_id"] / "manifest.json").resolve()
    if bundle_manifest != expected_bundle_manifest or bundle_manifest.read_bytes() != pointer_bytes:
        raise ValueError("Notebook 08 pointer and immutable report manifest differ")

    required_tables = {"main_comparison", "phase_winners", "paired_deltas_vs_dgd", "ablation_selection", "findings"}
    required_blobs = {"findings_markdown", "f1_by_phase", "roc_auc_by_phase"}
    if not required_tables <= set(manifest.get("output_schemas", {})):
        raise ValueError("Notebook 08 report is missing required CSV schemas")
    if not (required_tables | required_blobs) <= set(manifest.get("artifacts", {})):
        raise ValueError("Notebook 08 report is missing required artifacts")
    tables = {name: read_csv_artifact(root, manifest, name) for name in required_tables}
    optional_tables = {"egd_vs_emerg_paired_deltas"}
    for name in optional_tables:
        if name in manifest.get("output_schemas", {}) and name in manifest.get("artifacts", {}):
            tables[name] = read_csv_artifact(root, manifest, name)
    blobs = {}
    for name in required_blobs:
        artifact = manifest["artifacts"][name]
        path = resolve_inside(root, artifact["path"])
        if not path.is_file() or sha256_file(path) != artifact["sha256"]:
            raise ValueError(f"Report blob verification failed: {name}")
        blobs[name] = path
    return manifest, tables, blobs


def load_verified_protocol(root: Path, pointer: Path) -> tuple[dict[str, Any], dict[str, pd.DataFrame]]:
    pointer_bytes = pointer.read_bytes()
    manifest = strict_json_loads(pointer_bytes)
    if manifest.get("protocol_schema_version") != "ml1m-coldstart-v1" or manifest.get("protocol_status") != "PASS":
        raise ValueError("Unexpected notebook 02 protocol schema/status")
    checks = manifest.get("checks")
    if not isinstance(checks, list) or not checks or not all(isinstance(row, dict) and row.get("status") == "PASS" for row in checks):
        raise ValueError("Notebook 02 protocol checks are missing or not all PASS")
    bundle_manifest = resolve_inside(root, manifest["bundle_manifest"])
    expected_bundle_manifest = (pointer.parent / "generations" / manifest["bundle_id"] / "manifest.json").resolve()
    if bundle_manifest != expected_bundle_manifest or bundle_manifest.read_bytes() != pointer_bytes:
        raise ValueError("Notebook 02 pointer and immutable protocol manifest differ")
    required = {"tuning_train", "final_train", "validation_tasks", "evaluation_tasks", "users", "items", "item_cohorts"}
    if not required <= set(manifest.get("artifacts", {})) or not required <= set(manifest.get("output_schemas", {})):
        raise ValueError("Notebook 02 protocol is missing required demo tables")
    tables = {name: read_csv_artifact(root, manifest, name) for name in required}
    return manifest, tables


def load_first_valid(loader: Any, relative_manifest: Path, terminal_name: str) -> tuple[Path, Path, dict[str, Any], Any, list[str]]:
    errors: list[str] = []
    for root, pointer in artifact_candidates(relative_manifest, terminal_name):
        try:
            result = loader(root, pointer)
            if len(result) == 2:
                manifest, tables = result
                extra = None
            else:
                manifest, tables, extra = result
            return root, pointer, manifest, (tables, extra), errors
        except Exception as error:
            errors.append(f"{pointer}: {type(error).__name__}: {error}")
    raise RuntimeError(f"No verified artifact bundle found for {relative_manifest}. " + " | ".join(errors))


REPORT_ROOT, REPORT_POINTER, REPORT_MANIFEST, report_payload, REPORT_LOAD_ERRORS = load_first_valid(
    load_verified_report, REPORT_RELATIVE_MANIFEST, "results-v1"
)
REPORT_TABLES, REPORT_BLOBS = report_payload
PROTOCOL_ROOT, PROTOCOL_POINTER, PROTOCOL_MANIFEST, protocol_payload, PROTOCOL_LOAD_ERRORS = load_first_valid(
    load_verified_protocol, PROTOCOL_RELATIVE_MANIFEST, "coldstart-v1"
)
PROTOCOL_TABLES, _ = protocol_payload

REPORT_POINTER_SHA256 = sha256_file(REPORT_POINTER)
PROTOCOL_POINTER_SHA256 = sha256_file(PROTOCOL_POINTER)
RUNTIME_WARNINGS = []
if not PYG_AVAILABLE:
    RUNTIME_WARNINGS.append("torch_geometric is not installed; graph sample is exported as PyG-compatible tensors/dict")

show_records(
    [
        {
            "execution_context": EXECUTION_CONTEXT,
            "python": platform.python_version(),
            "torch_available": TORCH_AVAILABLE,
            "pyg_available": PYG_AVAILABLE,
            "report_bundle": REPORT_MANIFEST["bundle_id"],
            "report_status": REPORT_MANIFEST["report_status"],
            "protocol_bundle": PROTOCOL_MANIFEST["bundle_id"],
            "runtime_warnings": len(RUNTIME_WARNINGS),
        }
    ]
)

## Data and Protocol Walkthrough

Notebook 09 uses notebook 02 artifacts directly for a small demo view. This is
the right place to show dataset shape and phase visibility; it is not the right
place to rebuild MovieLens-1M splits.

In [ ]:
TUNING_TRAIN = PROTOCOL_TABLES["tuning_train"]
FINAL_TRAIN = PROTOCOL_TABLES["final_train"]
VALIDATION_TASKS = PROTOCOL_TABLES["validation_tasks"]
EVALUATION_TASKS = PROTOCOL_TABLES["evaluation_tasks"]
USERS = PROTOCOL_TABLES["users"]
ITEMS = PROTOCOL_TABLES["items"]
ITEM_COHORTS = PROTOCOL_TABLES["item_cohorts"]

PROTOCOL_SUMMARY = pd.DataFrame(
    [
        {"quantity": "users", "value": int(len(USERS))},
        {"quantity": "items", "value": int(len(ITEMS))},
        {"quantity": "old/final train rows", "value": int(len(FINAL_TRAIN))},
        {"quantity": "tuning train rows", "value": int(len(TUNING_TRAIN))},
        {"quantity": "validation task rows", "value": int(len(VALIDATION_TASKS))},
        {"quantity": "evaluation task rows", "value": int(len(EVALUATION_TASKS))},
        {"quantity": "evaluation query rows", "value": int(EVALUATION_TASKS["role"].astype(str).eq("query").sum())},
        {"quantity": "evaluation query positives", "value": int(EVALUATION_TASKS.loc[EVALUATION_TASKS["role"].astype(str).eq("query"), "label"].sum())},
    ]
)

support_roles = {
    "Cold": [],
    "Warm A": ["warm_a"],
    "Warm B": ["warm_a", "warm_b"],
    "Warm C": ["warm_a", "warm_b", "warm_c"],
}
PHASE_WALKTHROUGH = pd.DataFrame(
    [
        {
            "phase": phase,
            "visible_support_roles": ",".join(roles) if roles else "none",
            "support_rows": int(EVALUATION_TASKS["role"].astype(str).isin(roles).sum()),
            "query_rows": int(EVALUATION_TASKS["role"].astype(str).eq("query").sum()),
            "graph_edges_for_demo_policy": int(len(FINAL_TRAIN) + EVALUATION_TASKS["role"].astype(str).isin(roles).sum()),
        }
        for phase, roles in support_roles.items()
    ]
)
DEMO_ITEM_ID = int(EVALUATION_TASKS.loc[EVALUATION_TASKS["role"].astype(str).eq("query"), "item_id"].iloc[0])
DEMO_ITEM_ROWS = EVALUATION_TASKS[EVALUATION_TASKS["item_id"].astype(int).eq(DEMO_ITEM_ID)].copy()
DEMO_ITEM_ROWS = DEMO_ITEM_ROWS.sort_values(["item_rank", "source_row"]).head(68)
DEMO_ITEM_CONTEXT = pd.DataFrame(
    [
        {
            "demo_item_id": DEMO_ITEM_ID,
            "rows_shown": int(len(DEMO_ITEM_ROWS)),
            "warm_a_rows": int(DEMO_ITEM_ROWS["role"].astype(str).eq("warm_a").sum()),
            "warm_b_rows": int(DEMO_ITEM_ROWS["role"].astype(str).eq("warm_b").sum()),
            "warm_c_rows": int(DEMO_ITEM_ROWS["role"].astype(str).eq("warm_c").sum()),
            "query_rows_in_preview": int(DEMO_ITEM_ROWS["role"].astype(str).eq("query").sum()),
        }
    ]
)

display(PROTOCOL_SUMMARY)
display(PHASE_WALKTHROUGH)
display(DEMO_ITEM_CONTEXT)
display(DEMO_ITEM_ROWS[["source_row", "user_id", "user_idx", "item_id", "item_idx", "role", "label"]].head(12))

## PyG-Compatible Graph Walkthrough

The demo graph uses a tiny bounded subset: 64 final-train edges plus the demo
item's support rows. That is enough to show the graph handoff shape while
keeping runtime independent from full model training.

In [ ]:
base_edges = FINAL_TRAIN[["user_idx", "item_idx", "label"]].head(64).copy()
base_edges["edge_role"] = "final_train"
support_edges = DEMO_ITEM_ROWS.loc[
    DEMO_ITEM_ROWS["role"].astype(str).isin(["warm_a", "warm_b", "warm_c"]),
    ["user_idx", "item_idx", "label", "role"],
].copy()
support_edges = support_edges.rename(columns={"role": "edge_role"})
GRAPH_EDGE_SAMPLE = pd.concat([base_edges, support_edges], ignore_index=True)
GRAPH_EDGE_SAMPLE["user_idx"] = GRAPH_EDGE_SAMPLE["user_idx"].astype(int)
GRAPH_EDGE_SAMPLE["item_idx"] = GRAPH_EDGE_SAMPLE["item_idx"].astype(int)
GRAPH_EDGE_SAMPLE["label"] = GRAPH_EDGE_SAMPLE["label"].astype(int)

user_nodes = sorted(GRAPH_EDGE_SAMPLE["user_idx"].unique())
item_nodes = sorted(GRAPH_EDGE_SAMPLE["item_idx"].unique())
user_to_local = {user: index for index, user in enumerate(user_nodes)}
item_to_local = {item: index for index, item in enumerate(item_nodes)}
edge_index_np = np.array(
    [
        [user_to_local[int(user)] for user in GRAPH_EDGE_SAMPLE["user_idx"]],
        [item_to_local[int(item)] for item in GRAPH_EDGE_SAMPLE["item_idx"]],
    ],
    dtype=np.int64,
)
edge_label_np = GRAPH_EDGE_SAMPLE["label"].to_numpy(dtype=np.int64)

GRAPH_OBJECT = None
if TORCH_AVAILABLE:
    edge_index = torch.as_tensor(edge_index_np, dtype=torch.long)
    edge_label = torch.as_tensor(edge_label_np, dtype=torch.long)
    if PYG_AVAILABLE:
        GRAPH_OBJECT = HeteroData()
        GRAPH_OBJECT["user"].num_nodes = len(user_nodes)
        GRAPH_OBJECT["item"].num_nodes = len(item_nodes)
        GRAPH_OBJECT["user", "interacts", "item"].edge_index = edge_index
        GRAPH_OBJECT["user", "interacts", "item"].edge_label = edge_label
        graph_object_type = "torch_geometric.data.HeteroData"
    else:
        GRAPH_OBJECT = {"edge_index": edge_index, "edge_label": edge_label}
        graph_object_type = "torch tensor dict"
else:
    GRAPH_OBJECT = {"edge_index": edge_index_np, "edge_label": edge_label_np}
    graph_object_type = "numpy dict"

GRAPH_SUMMARY = pd.DataFrame(
    [
        {
            "graph_object_type": graph_object_type,
            "pyg_available": PYG_AVAILABLE,
            "users": len(user_nodes),
            "items": len(item_nodes),
            "edges": int(edge_index_np.shape[1]),
            "positive_edges": int(edge_label_np.sum()),
            "demo_item_in_graph": int(DEMO_ITEM_ROWS["item_idx"].iloc[0]) in item_to_local,
        }
    ]
)
GRAPH_EDGE_PREVIEW = GRAPH_EDGE_SAMPLE.head(12).copy()

display(GRAPH_SUMMARY)
display(GRAPH_EDGE_PREVIEW)
if PYG_AVAILABLE:
    display(Markdown(f"`HeteroData` created: `{GRAPH_OBJECT}`"))
else:
    display(Markdown("PyG is unavailable here; exported tensors keep the same `edge_index`/`edge_label` contract."))

## Model and Result Walkthrough

The model story is read from the frozen report evidence. Checkpoints are not
loaded here because notebooks 03-06 and 052 already validated training/evaluation
and notebook 09 must remain a bounded demo. EGD checkpoint structure is verified
inline for traceability.

In [ ]:
MAIN_COMPARISON = REPORT_TABLES["main_comparison"].copy()
PHASE_WINNERS = REPORT_TABLES["phase_winners"].copy()
PAIRED_DELTAS = REPORT_TABLES["paired_deltas_vs_dgd"].copy()
ABLATION_SELECTION = REPORT_TABLES["ablation_selection"].copy()
FINDINGS = REPORT_TABLES["findings"].copy()

MODEL_WALKTHROUGH = pd.DataFrame(
    [
        {
            "model": "LightGCN",
            "demo_role": "ID-only collaborative-filtering baseline",
            "source": "notebook 03 via notebook 07/08",
            "live_inference_here": False,
        },
        {
            "model": "EmerG",
            "demo_role": "side-feature graph-generation baseline with warm local adaptation",
            "source": "notebook 04 via notebook 07/08",
            "live_inference_here": False,
        },
        {
            "model": "EGD",
            "demo_role": "EmerG with conditional graph diffusion (v2)",
            "source": "notebook 052 via notebook 07/08",
            "live_inference_here": False,
        },
        {
            "model": "DGD",
            "demo_role": "differentiable graph diffusion with entmax sparsifier",
            "source": "notebook 05 via notebook 07/08",
            "live_inference_here": False,
        },
        {
            "model": "DGD ablation",
            "demo_role": "validation-selected component ablation per seed",
            "source": "notebook 06 via notebook 07/08",
            "live_inference_here": False,
        },
    ]
)
RESULT_SNAPSHOT = MAIN_COMPARISON.sort_values(["phase", "f1_mean"], ascending=[True, False])[
    ["model_label", "phase", "seed_count", "f1_report", "roc_auc_report"]
].reset_index(drop=True)
TOP_PHASE_WINNERS = PHASE_WINNERS[["phase", "model_label", "f1_mean", "roc_auc_mean"]].copy()

display(MODEL_WALKTHROUGH)
display(RESULT_SNAPSHOT)
display(TOP_PHASE_WINNERS)
display(ABLATION_SELECTION)
for figure_name in ("f1_by_phase", "roc_auc_by_phase"):
    path = REPORT_BLOBS.get(figure_name)
    if path and path.is_file():
        display(Image(filename=str(path)))

## Takeaways and Demo Export

Export only compact demo artifacts: protocol summary, graph preview, result
snapshot, model walkthrough, and a manifest. This gives presentations a single
handoff without duplicating the full dataset or model outputs.

In [ ]:
findings_markdown = REPORT_BLOBS["findings_markdown"].read_text(encoding="utf-8")
DEMO_TAKEAWAYS = "\n".join(
    [
        "# End-to-End Demo Takeaways",
        "",
        "- Notebook 02 defines the leakage-safe MovieLens-1M cold-start protocol.",
        "- Notebooks 03-06 and 052 train/evaluate models and export immutable per-seed bundles.",
        "- Notebook 07 aggregates the exact target seeds and preserves WARN/PARTIAL status.",
        "- Notebook 08 formats final tables and figures for reporting.",
        "- Notebook 09 demonstrates the handoff and graph shape without retraining.",
        "",
        "## Report Findings",
        findings_markdown.strip(),
    ]
)

DEMO_CHECKS: list[dict[str, Any]] = []


def demo_check(name: str, condition: bool, observed: Any, expected: Any, severity: str = "ERROR") -> None:
    DEMO_CHECKS.append(
        {
            "check": name,
            "status": "PASS" if condition else severity,
            "observed": observed,
            "expected": expected,
            "severity": severity,
        }
    )


demo_check("report schema", REPORT_MANIFEST["report_schema_version"] == "results-report-v1", REPORT_MANIFEST["report_schema_version"], "results-report-v1")
demo_check("protocol schema", PROTOCOL_MANIFEST["protocol_schema_version"] == "ml1m-coldstart-v1", PROTOCOL_MANIFEST["protocol_schema_version"], "ml1m-coldstart-v1")
demo_check("graph sample nonempty", int(edge_index_np.shape[1]) > 0, int(edge_index_np.shape[1]), "> 0")
demo_check("result snapshot nonempty", not RESULT_SNAPSHOT.empty, len(RESULT_SNAPSHOT), "> 0")
demo_check("PyG runtime available", PYG_AVAILABLE, PYG_AVAILABLE, True, severity="WARN")
demo_check("upstream report clean", REPORT_MANIFEST["report_status"] == "PASS", REPORT_MANIFEST["report_status"], "PASS", severity="WARN")
demo_check("upstream coverage complete", REPORT_MANIFEST["coverage_status"] == "COMPLETE", REPORT_MANIFEST["coverage_status"], "COMPLETE", severity="WARN")
demo_check("EGD model present in report", not MAIN_COMPARISON.query("model_label == 'EGD'").empty, bool(not MAIN_COMPARISON.query("model_label == 'EGD'").empty), True, severity="WARN")

DEMO_STATUS = (
    "BLOCKED"
    if any(row["status"] == "ERROR" for row in DEMO_CHECKS)
    else "WARN"
    if any(row["status"] == "WARN" for row in DEMO_CHECKS)
    else "PASS"
)
if DEMO_STATUS == "BLOCKED":
    display(pd.DataFrame(DEMO_CHECKS))
    raise RuntimeError("Notebook 09 demo export is blocked by failed checks")


def json_ready(value: Any) -> Any:
    if value is None or isinstance(value, (str, int, bool)):
        return value
    if isinstance(value, float):
        if not np.isfinite(value):
            raise ValueError(f"Non-finite value cannot be serialized to JSON: {value}")
        return value
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_ready(item) for item in value]
    if hasattr(value, "item"):
        return json_ready(value.item())
    return str(value)


def write_text(path: Path, content: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    temporary.write_text(content, encoding="utf-8")
    temporary.replace(path)


def write_csv(path: Path, table: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    table.to_csv(temporary, index=False, lineterminator="\n")
    temporary.replace(path)


def relative_output(path: Path) -> str:
    return str(path.resolve().relative_to(ARTIFACT_ROOT.resolve()))


def csv_schema(table: pd.DataFrame) -> dict[str, Any]:
    def dtype_name(dtype: Any) -> str:
        name = str(dtype)
        return "string" if name in {"str", "string"} or name.startswith("string") else name

    return {
        "columns": list(table.columns),
        "read_csv_dtypes": {column: dtype_name(dtype) for column, dtype in table.dtypes.items()},
    }


DEMO_TABLES = {
    "protocol_summary": PROTOCOL_SUMMARY,
    "phase_walkthrough": PHASE_WALKTHROUGH,
    "demo_item_context": DEMO_ITEM_CONTEXT,
    "graph_edge_sample": GRAPH_EDGE_SAMPLE,
    "graph_summary": GRAPH_SUMMARY,
    "model_walkthrough": MODEL_WALKTHROUGH,
    "result_snapshot": RESULT_SNAPSHOT,
    "phase_winners": TOP_PHASE_WINNERS,
}
bundle_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + "-" + uuid.uuid4().hex[:12]
staging_root = OUTPUT_ROOT / f".staging-{bundle_id}"
generation_root = OUTPUT_ROOT / "generations" / bundle_id
pointer_path = OUTPUT_ROOT / "manifest.json"
previous_pointer_text = pointer_path.read_text(encoding="utf-8") if pointer_path.is_file() else None
generation_published = False
pointer_write_attempted = False
staging_root.mkdir(parents=True, exist_ok=False)
OUTPUT_ARTIFACTS: dict[str, Any] = {}
OUTPUT_SCHEMAS: dict[str, Any] = {}

try:
    for name, table in DEMO_TABLES.items():
        staging_path = staging_root / f"{name}.csv"
        published_path = generation_root / f"{name}.csv"
        write_csv(staging_path, table)
        OUTPUT_ARTIFACTS[name] = {
            "path": relative_output(published_path),
            "sha256": sha256_file(staging_path),
            "rows": len(table),
        }
        OUTPUT_SCHEMAS[name] = csv_schema(table)

    takeaway_path = staging_root / "demo_takeaways.md"
    write_text(takeaway_path, DEMO_TAKEAWAYS + "\n")
    OUTPUT_ARTIFACTS["demo_takeaways"] = {
        "path": relative_output(generation_root / "demo_takeaways.md"),
        "sha256": sha256_file(takeaway_path),
    }

    graph_payload = {
        "edge_index": edge_index_np.tolist(),
        "edge_label": edge_label_np.tolist(),
        "global_user_idx": user_nodes,
        "global_item_idx": item_nodes,
        "edge_roles": GRAPH_EDGE_SAMPLE["edge_role"].astype(str).tolist(),
    }
    graph_payload_path = staging_root / "pyg_graph_sample.json"
    write_text(graph_payload_path, strict_json_dumps(json_ready(graph_payload), indent=2, sort_keys=True) + "\n")
    OUTPUT_ARTIFACTS["pyg_graph_sample"] = {
        "path": relative_output(generation_root / "pyg_graph_sample.json"),
        "sha256": sha256_file(graph_payload_path),
    }

    manifest = {
        "demo_schema_version": "end-to-end-demo-v1",
        "demo_status": DEMO_STATUS,
        "bundle_id": bundle_id,
        "bundle_manifest": relative_output(generation_root / "manifest.json"),
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "upstream_report": {
            "schema_version": REPORT_MANIFEST["report_schema_version"],
            "bundle_id": REPORT_MANIFEST["bundle_id"],
            "pointer_sha256": REPORT_POINTER_SHA256,
            "report_status": REPORT_MANIFEST["report_status"],
            "coverage_status": REPORT_MANIFEST["coverage_status"],
        },
        "upstream_protocol": {
            "schema_version": PROTOCOL_MANIFEST["protocol_schema_version"],
            "bundle_id": PROTOCOL_MANIFEST["bundle_id"],
            "pointer_sha256": PROTOCOL_POINTER_SHA256,
        },
        "runtime": {
            "torch_available": TORCH_AVAILABLE,
            "pyg_available": PYG_AVAILABLE,
            "graph_object_type": graph_object_type,
            "warnings": RUNTIME_WARNINGS,
        },
        "summary": {
            "demo_item_id": DEMO_ITEM_ID,
            "graph_edges": int(edge_index_np.shape[1]),
            "result_rows": int(len(RESULT_SNAPSHOT)),
            "report_best_model": REPORT_MANIFEST["summary"]["best_model_by_mean_f1"],
        },
        "checks": DEMO_CHECKS,
        "artifacts": OUTPUT_ARTIFACTS,
        "output_schemas": OUTPUT_SCHEMAS,
        "demo_contract": {
            "training": "none",
            "dataset": "read-only verified notebook-02 protocol artifacts",
            "results": "read-only verified notebook-08 report artifacts (sourced from notebooks 03-06, 052)",
            "pyg": "HeteroData when torch_geometric is installed; otherwise exported PyG-compatible edge tensors",
        },
    }

    manifest_text = strict_json_dumps(json_ready(manifest), indent=2, sort_keys=True) + "\n"
    write_text(staging_root / "manifest.json", manifest_text)
    generation_root.parent.mkdir(parents=True, exist_ok=True)
    staging_root.replace(generation_root)
    generation_published = True
    artifact_hashes_match = all(
        sha256_file(resolve_inside(ARTIFACT_ROOT, artifact["path"])) == artifact["sha256"]
        for artifact in OUTPUT_ARTIFACTS.values()
    )
    if not artifact_hashes_match or (generation_root / "manifest.json").read_text(encoding="utf-8") != manifest_text:
        raise RuntimeError("Published demo generation failed pre-pointer verification")
    pointer_write_attempted = True
    write_text(pointer_path, manifest_text)
    if pointer_path.read_text(encoding="utf-8") != manifest_text:
        raise RuntimeError("Demo pointer does not match the verified generation")
except Exception:
    try:
        if pointer_write_attempted:
            if previous_pointer_text is None:
                pointer_path.unlink(missing_ok=True)
            else:
                write_text(pointer_path, previous_pointer_text)
    finally:
        if staging_root.exists():
            shutil.rmtree(staging_root, ignore_errors=True)
        if generation_published and generation_root.exists():
            shutil.rmtree(generation_root)
    raise

DEMO_MANIFEST = manifest
DEMO_POINTER = pointer_path
show_records(
    [
        {
            "demo_status": DEMO_STATUS,
            "bundle_id": bundle_id,
            "manifest": str(pointer_path),
            "artifacts": len(OUTPUT_ARTIFACTS),
            "graph_object_type": graph_object_type,
        }
    ]
)

## Final Verification

This is the terminal notebook in the pipeline. A clean demo reports `READY`;
a usable demo with optional-runtime or upstream warnings reports `WARN`;
structural failures report `BLOCKED` and raise.

In [ ]:
FINAL_CHECKS: list[dict[str, Any]] = list(DEMO_CHECKS)


def final_check(name: str, condition: bool, observed: Any, expected: Any) -> None:
    FINAL_CHECKS.append(
        {
            "check": name,
            "status": "PASS" if condition else "ERROR",
            "observed": observed,
            "expected": expected,
            "severity": "ERROR",
        }
    )


pointer_matches_bundle = DEMO_POINTER.read_text(encoding="utf-8") == (generation_root / "manifest.json").read_text(encoding="utf-8")
all_hashes_verify = all(
    sha256_file(resolve_inside(ARTIFACT_ROOT, artifact["path"])) == artifact["sha256"]
    for artifact in DEMO_MANIFEST["artifacts"].values()
)
final_check("manifest pointer matches demo bundle", pointer_matches_bundle, pointer_matches_bundle, True)
final_check("demo artifact hashes verify", all_hashes_verify, all_hashes_verify, True)
final_check("graph payload exported", "pyg_graph_sample" in DEMO_MANIFEST["artifacts"], True, True)
final_check("takeaways exported", "demo_takeaways" in DEMO_MANIFEST["artifacts"], True, True)

FINAL_AUDIT = pd.DataFrame(FINAL_CHECKS)
FINAL_BLOCKED = FINAL_AUDIT["status"].eq("ERROR").any()
NOTEBOOK_STATUS = "BLOCKED" if FINAL_BLOCKED else "WARN" if DEMO_STATUS == "WARN" else "READY"
display(FINAL_AUDIT)
display(Markdown("### Notebook 09 end-to-end demo: " + NOTEBOOK_STATUS))

if FINAL_BLOCKED:
    raise RuntimeError("Notebook 09 final demo checks failed")

display(
    Markdown(
        "**Pipeline complete:** use `demos/ml-1m/end-to-end-v1/manifest.json` for a compact demo handoff, "
        "or go back to notebooks 02, 07, and 08 for full protocol, evaluation, and report details."
    )
)